# E7 — Reinterpreting experience by discovering overlapping behavioral concepts
CPU only. The learner sees object IDs, actions and binary outcomes, not underlying roles. Rather than one permanent object label, infer an uncertain partition of objects for each discovered group of actions. Every new observation can change the posterior meaning of old observations. This is exact Bayesian structure learning, **not a novel algorithm** or full Pāṇinian learning. Comparators: identity replay, independent-action latent clustering, shared latent clustering. Related: [hierarchical latent structure learning](https://pmc.ncbi.nlm.nih.gov/articles/PMC13419619/), [OPINE-World](https://arxiv.org/abs/2607.01531), [reward-predictive abstraction](https://doi.org/10.1371/journal.pcbi.1008317). Predictions: a shared abstraction helps aligned mechanics; overlapping concepts help crossed mechanics; identity is hard to beat without structure.

In [ ]:
import numpy as np
from scipy.special import betaln, logsumexp, gammaln
from time import perf_counter

def partitions(n):
    def rec(p):
        if len(p)==n:
            yield p; return
        for k in range(max(p)+2): yield from rec(p+[k])
    return np.array(list(rec([0])))

def crp(p,alpha=1.):
    sizes=np.eye(p.shape[1])[p].sum(1)
    return ((sizes>0).sum(1)*np.log(alpha)
            +np.where(sizes>0,gammaln(np.maximum(sizes,1)),0).sum(1)
            -gammaln(alpha+p.shape[1])+gammaln(alpha))

N,A=7,4
P,Q=partitions(N),partitions(A)
M=(P[:,:,None]==P[:,None,:]).astype(float)
first=np.array([[list(row).index(k)==i for i,k in enumerate(row)] for row in P])
logp,logq=crp(P),crp(Q)
identity_idx=np.flatnonzero((P==np.arange(N)).all(1))[0]
robust_logp=np.logaddexp(logp+np.log(.5),np.where(np.arange(len(P))==identity_idx,np.log(.5),-np.inf))
subsets=[tuple(i for i in range(A) if mask>>i&1) for mask in range(1,1<<A)]
lookup={s:i for i,s in enumerate(subsets)}
B=np.zeros((len(Q),len(subsets)))
for i,q in enumerate(Q):
    for k in range(q.max()+1): B[i,lookup[tuple(np.flatnonzero(q==k))]]=1
assert len(P)==877 and len(Q)==15
assert all(np.isclose(np.exp(p).sum(),1) for p in (logp,logq,robust_logp))

Each candidate meaning groups observations and pools outcomes; exact Beta-Bernoulli marginal likelihood decides how much to trust it. Integrating all partitions is a strong full-history baseline. The adaptive model also marginalizes over which actions share a partition. No hard labels or future outcomes. Complexity prior penalizes proliferating concepts, not a hand-selected threshold.

In [ ]:
def infer(success,total,beta=1.):
    s,n=M@success,M@total
    loglik=((betaln(beta+s,beta+n-s)-betaln(beta,beta))*first[:,:,None]).sum(1)
    mean=(beta+s)/(2*beta+n)
    def structures(prior):
        evidence,predictions=[],[]
        for a in subsets:
            z=prior+loglik[:,a].sum(1); e=logsumexp(z)
            evidence.append(e)
            predictions.append(np.einsum('p,pia->ia',np.exp(z-e),mean))
        w=logq+B@np.array(evidence); w=np.exp(w-logsumexp(w)); marginal=w@B
        adaptive=np.stack([sum(marginal[j]*predictions[j][:,a] for j,b in enumerate(subsets) if a in b) for a in range(A)],axis=1)
        return predictions,adaptive
    predictions,adaptive=structures(logp)
    _,robust=structures(robust_logp)
    shared=predictions[lookup[tuple(range(A))]]
    independent=np.stack([predictions[lookup[(a,)]][:,a] for a in range(A)],axis=1)
    identity=(beta+success)/(2*beta+total)
    return np.array([identity,independent,shared,adaptive,robust])

assert np.allclose(infer(np.zeros((N,A)),np.zeros((N,A))),.5)

def world(rng,mode):
    roles=rng.permutation([0,0,0,1,1,1,1]); other=rng.permutation([0,0,0,1,1,1,1])
    if mode=='shared': z=np.stack([roles]*A,axis=1)
    elif mode=='crossed': z=np.stack([roles,roles,other,other],axis=1)
    else: z=rng.integers(0,2,(N,A))
    z=z[:,rng.permutation(A)]^rng.integers(0,2,A)
    return .05+.9*z

Predict probabilities after 8,16,28,56 noisy observations, score expected squared error to true response probabilities (Brier excess; lower is better). This evaluates all object-action pairs including previously observed ones; not a held-out-object test. All arms see the same observations. The generator has strong near-deterministic structure. Beta=.25 and an identity-prior mixture are exploratory improvements selected after the initial run. Replications use fresh seeds, same generator. No GPU.

In [ ]:
def run(seed,beta,episodes=100):
    rng=np.random.default_rng(seed)
    for mode in ('shared','crossed','random'):
        scores=[]
        for _ in range(episodes):
            truth=world(rng,mode); s=np.zeros((N,A)); n=s.copy(); obs=[]
            for repeat in range(3): obs.extend(rng.permutation(N*A))
            rows=[]
            for t,x in enumerate(obs[:56],1):
                i,a=divmod(x,A); n[i,a]+=1; s[i,a]+=rng.random()<truth[i,a]
                if t in (8,16,28,56):
                    rows.append(((infer(s,n,beta)-truth)**2).mean((1,2)))
            scores.append(rows)
        v=np.array(scores)
        print(seed,mode,'rows 8,16,28,56; columns identity,independent,shared,adaptive,robust')
        print(np.round(v.mean(0),4))
        d=v[:,2,2]-v[:,2,3]
        boot=np.random.default_rng(22).choice(d,(2000,len(d)),replace=True).mean(1)
        print('adaptive improvement over shared at 28:',round(d.mean(),4),np.round(np.quantile(boot,[.025,.975]),4))

t=perf_counter()
for seed in (42,143): run(seed,1.)
for seed in (211,397): run(seed,.25)
print('seconds',perf_counter()-t)

## Actual standalone CPU measurements
Original Beta(1,1) model, seed 42: at 28 observations shared vs adaptive on shared mechanics .0521 vs .0661; crossed .1053 vs .1129; random .1168 vs .1173. Identity replay beat both on crossed/random at this point. At 56 crossed adaptive .0400 vs shared .0485. Seed 143 confirmed direction.

Revised Beta(.25,.25), fresh seeds 211/397, means across the two seed runs:
| 56 observations | Identity | Shared | Adaptive | Adaptive + identity prior |
|---|---:|---:|---:|---:|
| Shared | .0215 | .0073 | .0078 | .0083 |
| Crossed | .0234 | .0195 | .0174 | .0210 |
| Random | .0233 | .0318 | .0274 | .0236 |

At 28 crossed, adaptive improved over shared by .0010 and .0018 in the two seeds; bootstrap intervals [.0003,.0016] and [.0011,.0024]. No uniform win over identity replay; sparse data and wrong abstraction are costly. The revised prior also improves the simple baseline; do not attribute that benefit to semantic recursion. No novel SOTA claim. The adaptive model is conventional exact Bayesian co-clustering with a partition-of-partitions. It exhaustively enumerates Bell-number-many hypotheses, cannot scale directly, and discovers no relations beyond supplied object/action identities. Next test: discover relational abstractions themselves (same/different, role exchange, composition), use inferred relational evidence to re-encode raw experience, compare against full-history Bayesian relational inference and a small neural model on genuinely held-out objects and tasks. If only ordinary posterior inference is responsible, reject the claimed extra mechanism.